# YemenVision-MTMC — Colab Data Acquisition & Training Pipeline

**الغرض:** تنزيل مجموعات البيانات التي يمكن الوصول إليها قانونيًا، فحصها، تجهيزها، تدريب كشف المركبات وReID، تشغيل التتبع والربط متعدد الكاميرات عند توفر بياناته، وحفظ النتائج في Google Drive.

> ملاحظة حاسمة: ليست كل البيانات متاحة للتنزيل العام. CityFlow يتطلب قبول اتفاقية AI City، وVeRi-776 يُطلب من أصحابه عبر البريد لأغراض بحثية غير تجارية، وVehicleID يتطلب المرور ببوابة مصدره. لا توجد خلية تستطيع تجاوز هذه الموافقات. هذه الدفترية تنزّل تلقائيًا UA-DETRAC من روابط الملفات المنشورة، وتتيح إدخال روابط الملفات المصرح بها لبقية المجموعات، ثم تتحقق من SHA256 إذا زُوّدت به.

## 0. اختيار GPU

In [ ]:
import os, sys, subprocess, json, hashlib, zipfile, tarfile, shutil, requests, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("فعّل GPU من Runtime > Change runtime type > T4 GPU ثم أعد تشغيل الخلية.")
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

## 1. ربط Drive واستنساخ المشروع

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC')
ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/Al-Nahari/YemenVision-MTMC.git'
REPO = ROOT / 'YemenVision-MTMC'

if not REPO.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=False)

RESEARCH = REPO / 'research'
if not RESEARCH.exists():
    found = list(REPO.rglob('research'))
    if not found: raise FileNotFoundError('لم أعثر على مجلد research داخل المستودع.')
    RESEARCH = found[0]

DATA = ROOT / 'datasets'
RAW = DATA / 'raw'
PREP = DATA / 'prepared'
RUNS = ROOT / 'runs'
ARTIFACTS = ROOT / 'artifacts'
for p in [RAW, PREP, RUNS, ARTIFACTS]: p.mkdir(parents=True, exist_ok=True)
print('REPO=',REPO,'RESEARCH=',RESEARCH,'DATA=',DATA)

## 2. تثبيت المتطلبات

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','pip'],check=True)
packages = [
 'ultralytics','timm','opencv-python-headless','pandas','numpy','scipy',
 'scikit-learn','matplotlib','seaborn','pyyaml','tqdm','gdown',
 'lxml','Pillow','tensorboard','motmetrics','filterpy','lap'
]
subprocess.run([sys.executable,'-m','pip','install','-q']+packages,check=True)
print('Dependencies installed')

## 3. مصادر البيانات الرسمية وما الذي يمكن تنزيله

| البيانات | المهمة | حالة الوصول |
|---|---|---|
| [CityFlowV2 / AI City 2022](https://www.aicitychallenge.org/2022-data-and-evaluation/) | MTMC وفيديو متعدد الكاميرات مع IDs | صفحة التنزيل تنص على قبول اتفاقية الترخيص بالنقر |
| [VeRi-776](https://vehiclereid.github.io/VeRi/) | Vehicle ReID متعدد الكاميرات | طلب وصول بالبريد، استخدام غير تجاري |
| [VehicleID](https://www.pkuml.org/resources/pku-vehicleid.html) | Vehicle ReID | عبر صفحة المصدر/طلب الوصول |
| [UA-DETRAC](https://www.detrac-db.rit.albany.edu/) | كشف وتتبع فيديو كاميرا واحدة | روابط ملفات منشورة أدناه؛ تحقق من توفرها |

CityFlow هو الأهم لمهمة الربط بين الكاميرات. UA-DETRAC لا يعوض CityFlow لأنه لا يوفر نفس بيانات الهوية المتزامنة متعددة الكاميرات. لا تخلط مجموعات ReID مع فيديوهات MTMC.

## 4. أدوات تنزيل قوية مع إعادة المحاولة وفحص الملف

In [ ]:
from urllib.parse import urlparse

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def download_file(url, out, expected_sha256=None, retries=4):
    out=Path(out); out.parent.mkdir(parents=True,exist_ok=True)
    if not url or urlparse(url).scheme != 'https':
        raise ValueError('الرابط يجب أن يكون HTTPS مباشرًا للملف.')
    last=None
    for attempt in range(retries):
        try:
            with requests.get(url,stream=True,timeout=(30,180),headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                total=int(r.headers.get('content-length',0))
                tmp=out.with_suffix(out.suffix+'.part')
                got=0
                with open(tmp,'wb') as f:
                    for chunk in r.iter_content(8*1024*1024):
                        if chunk: f.write(chunk); got+=len(chunk)
                if got < 1024*100:
                    raise RuntimeError(f'الملف صغير بصورة غير منطقية ({got} bytes)، ربما صفحة تسجيل دخول وليست ملفًا.')
                tmp.replace(out)
            if expected_sha256:
                actual=sha256(out)
                if actual.lower()!=expected_sha256.lower():
                    out.unlink(missing_ok=True)
                    raise RuntimeError(f'SHA256 غير مطابق: {actual}')
            print('Downloaded:',out,'bytes=',out.stat().st_size,'sha256=',sha256(out))
            return out
        except Exception as e:
            last=e
            print(f'Attempt {attempt+1}/{retries} failed:',repr(e))
            time.sleep(3*(attempt+1))
    raise RuntimeError(f'فشل تنزيل {url}: {last}')

def safe_extract(archive, dest):
    archive=Path(archive); dest=Path(dest); dest.mkdir(parents=True,exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            bad=[n for n in z.namelist() if Path(n).is_absolute() or '..' in Path(n).parts]
            if bad: raise RuntimeError('Archive contains unsafe paths')
            z.extractall(dest)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            for m in t.getmembers():
                if Path(m.name).is_absolute() or '..' in Path(m.name).parts:
                    raise RuntimeError('Archive contains unsafe paths')
            t.extractall(dest)
    else:
        raise ValueError('صيغة الأرشيف غير مدعومة')
    print('Extracted to',dest)

## 5. تنزيل UA-DETRAC تلقائيًا

هذه روابط ملفات منشورة على نطاق المشروع الرسمي `detrac-db.rit.albany.edu`. قد تتغير أو تتوقف؛ الخلية تتحقق من الاستجابة وحجم الملف ولا تعتبر صفحة HTML تنزيلًا ناجحًا. حجمها كبير وقد تحتاج مساحة Drive ووقتًا.

In [ ]:
UA = RAW / 'UA_DETRAC'
UA.mkdir(parents=True, exist_ok=True)

UA_URLS = {
    'train_videos': 'https://detrac-db.rit.albany.edu/Data/DETRAC-train-data.zip',
    'test_videos':  'https://detrac-db.rit.albany.edu/Data/DETRAC-test-data.zip',
}
UA_FILES = {}
for key, url in UA_URLS.items():
    target = UA / (key + '.zip')
    if target.exists() and target.stat().st_size > 1024*1024:
        print(f'ملف {key} موجود مسبقاً في Drive بحجم {round(target.stat().st_size/1e9, 2)} GB.')
        UA_FILES[key] = target
        continue
    try:
        UA_FILES[key] = download_file(url, target)
    except Exception as e:
        print(f'تنبيه: تعذر التنزيل الآلي لـ {key} ({e}). خادم المصدر قد يكون محمياً بـ Cloudflare.')
        print(f'-> يرجى تنزيل الملف يدوياً عبر المتصفح من: {url}')
        print(f'-> ثم رفعه مباشرة إلى المجلد: {UA}')

for key, path_item in list(UA_FILES.items()):
    dest = UA / key
    flag = dest / '.extracted_ok'
    if not flag.exists():
        print(f'جاري فك ضغط {key} ...')
        safe_extract(path_item, dest)
        flag.write_text('OK')
    else:
        print(f'تم فك ضغط {key} مسبقاً في: {dest}')

print('UA-DETRAC files ready:', list(UA_FILES.keys()))

## 6. CityFlowV2 — تنزيل بعد قبول اتفاقية AI City

افتح صفحة المصدر، اقبل اتفاقية البيانات بنفسك، ثم انسخ رابط ملف Track 1 الذي يظهر لك (قد يكون رابطًا مؤقتًا/محميًا). لا تلصق رابط صفحة HTML؛ المطلوب رابط تنزيل الملف بعد المصادقة. إذا كان الرابط يتطلب جلسة متصفح، نزّل الملف من المصدر إلى جهازك/Drive ثم اكتب مساره في الخلية التالية.

In [ ]:
CITYFLOW_URL = ''       # الصق رابط الملف المباشر بعد قبول الاتفاقية
CITYFLOW_SHA256 = ''    # أدخل SHA256 إن نشره المصدر
CITYFLOW_LOCAL_FILE = ''# أو مسار ملف قمت بتنزيله قانونيًا إلى Drive

CITYFLOW_DIR = RAW / 'CityFlowV2'
CITYFLOW_DIR.mkdir(parents=True,exist_ok=True)
cityflow_archive=None

if CITYFLOW_LOCAL_FILE and Path(CITYFLOW_LOCAL_FILE).is_file():
    cityflow_archive=Path(CITYFLOW_LOCAL_FILE)
elif CITYFLOW_URL:
    cityflow_archive=download_file(CITYFLOW_URL,CITYFLOW_DIR/'cityflowv2.zip',CITYFLOW_SHA256 or None)
else:
    print('WAITING: CityFlowV2 يحتاج قبول الاتفاقية ثم إدخال رابط التنزيل أو مسار الملف.')

if cityflow_archive:
    safe_extract(cityflow_archive,CITYFLOW_DIR/'extracted')

## 7. VeRi-776 وVehicleID — طلب الوصول ثم إدخال الملف

مصدر VeRi يطلب إرسال الاسم والجهة إلى صاحب البيانات ويقصر التوزيع على البحث غير التجاري. لا توجد وصلة تنزيل عامة موثوقة ينبغي اختراعها. VehicleID أيضًا يُؤخذ من صفحة PKU الرسمية. بعد الحصول على ملفات الأرشيف من المصدر، ارفعها إلى Drive أو أدخل روابطها المصرح بها هنا.

In [ ]:
ACCESS_DATASETS = {
 'VeRi776': {
   'url':'', 'sha256':'', 'local_file':'',
   'source':'https://vehiclereid.github.io/VeRi/'
 },
 'VehicleID': {
   'url':'', 'sha256':'', 'local_file':'',
   'source':'https://www.pkuml.org/resources/pku-vehicleid.html'
 }
}
ACCESS_DIR=RAW/'ReID'
ACCESS_DIR.mkdir(parents=True,exist_ok=True)
ACCESS_RESULTS={}

for name,cfg in ACCESS_DATASETS.items():
    archive=None
    if cfg['local_file'] and Path(cfg['local_file']).is_file():
        archive=Path(cfg['local_file'])
    elif cfg['url']:
        archive=download_file(cfg['url'],ACCESS_DIR/(name+'.archive'),cfg['sha256'] or None)
    else:
        print(name,'WAITING FOR AUTHORIZED DOWNLOAD. Source:',cfg['source'])
    if archive:
        dest=ACCESS_DIR/name
        safe_extract(archive,dest)
        ACCESS_RESULTS[name]=dest
print('Ready ReID archives:',ACCESS_RESULTS)

## 8. فهرسة الملفات وفحص التنزيلات

In [ ]:
def inventory(root):
    rows = []
    for p in Path(root).rglob('*'):
        if p.is_file():
            rows.append((str(p), p.stat().st_size))
    return rows

files = inventory(RAW)
print('Total files in RAW:', len(files))
print('Total raw size GB:', round(sum(x[1] for x in files) / 1024**3, 3))
if files:
    print('Sample 10 files:')
    for row in files[:10]:
        print(' ', row[0], round(row[1]/1e6, 2), 'MB')
else:
    print('تنبيه: مجلد RAW فارغ حالياً. تأكد من تحميل الأرشيفات المطلوبة إلى Drive قبل بدء المعالجة.')

## 9. تجهيز بيانات الكشف

UA-DETRAC annotations بصيغة XML وليست YOLO TXT جاهزة. يجب تحويل XML إلى YOLO وفقًا لبنية الأرشيف الفعلية. لا تشغّل تدريب YOLO على مجلد صور بلا labels. الخلية التالية تعرض ملفات XML وتفحص توفرها؛ يجب إكمال المحول حسب بنية ملفات XML الموجودة في النسخة التي تم تنزيلها.

In [ ]:
# 9. تجهيز بيانات الكشف وتحويل XML إلى YOLO مع عزل التسرب
import subprocess

DET_PREP = PREP / 'detection'
DET_PREP.mkdir(parents=True, exist_ok=True)
YOLO_YAML = DET_PREP / 'data.yaml'

xml_files = list(RAW.rglob('*.xml'))
print(f'ملفات XML المعثور عليها: {len(xml_files)}')

if xml_files:
    convert_cli = RESEARCH / 'data_engineering' / 'cli' / 'convert_uadetrac_to_yolo.py'
    if convert_cli.exists():
        print('تشغيل محول UA-DETRAC الرسمي المانع للتسرب (مع أخذ عينة كل 5 إطارات)...')
        res = subprocess.run([
            sys.executable, str(convert_cli),
            '--raw-dir', str(RAW),
            '--output-dir', str(DET_PREP),
            '--sample-rate', '5'
        ], capture_output=True, text=True)
        print(res.stdout)
        if res.returncode != 0:
            print('خطأ في المحول:', res.stderr)
    else:
        print('ملف المحول غير موجود في المستودع.')
else:
    print('لا توجد ملفات XML في RAW. في حال الرغبة بالتشغيل السريع بدون انتظار، سنستخدم كاشف COCO المسبق.')

## 10. تدريب YOLO للكشف (يبدأ فقط عند اكتمال labels)

In [ ]:
from ultralytics import YOLO
import yaml

LOCAL_DET = Path('/content/detection')
EXP_DIR = RUNS / 'detection' / 'yolo11n_vehicle'
WEIGHTS_DIR = EXP_DIR / 'weights'
BEST_DETECTOR = WEIGHTS_DIR / 'best.pt'
cfg_file = RESEARCH / 'configs' / 'detection_yolo11n.yaml'

if not cfg_file.exists():
    raise FileNotFoundError(f'Config not found: {cfg_file}')
with open(cfg_file, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# نسخ سريع إلى /content لزيادة سرعة القراءة
LOCAL_DET.mkdir(parents=True, exist_ok=True)
flag_copied = LOCAL_DET / '.copy_done'
if (PREP / 'detection' / 'data.yaml').exists() and not flag_copied.exists():
    print('نسخ بيانات الكشف إلى /content ...')
    for folder in ['images', 'labels']:
        src = PREP / 'detection' / folder
        dst = LOCAL_DET / folder
        if src.exists() and not dst.exists():
            shutil.copytree(src, dst)
    shutil.copy2(PREP / 'detection' / 'data.yaml', LOCAL_DET / 'data.yaml')
    flag_copied.write_text('OK')

target_yaml = LOCAL_DET / 'data.yaml' if (LOCAL_DET / 'data.yaml').exists() else (PREP / 'detection' / 'data.yaml')

if target_yaml.exists():
    last_pt = WEIGHTS_DIR / 'last.pt'
    if last_pt.exists():
        print(f'استئناف التدريب من: {last_pt}')
        model = YOLO(str(last_pt))
        det_results = model.train(resume=True)
    else:
        print(f'بدء تدريب YOLO على بيانات المشروع باستخدام إعدادات {cfg_file.name} ...')
        model = YOLO(cfg.get('model', 'yolo11n.pt'))
        det_results = model.train(
            data=str(target_yaml),
            epochs=cfg.get('epochs', 100),
            imgsz=cfg.get('imgsz', 960),
            batch=16,
            optimizer=cfg.get('optimizer', 'AdamW'),
            lr0=cfg.get('lr0', 0.001),
            lrf=cfg.get('lrf', 0.01),
            warmup_epochs=cfg.get('warmup_epochs', 3),
            weight_decay=cfg.get('weight_decay', 0.0005),
            patience=cfg.get('patience', 20),
            seed=cfg.get('seed', 17),
            project=str(RUNS / 'detection'),
            name='yolo11n_vehicle',
            save=True,
            save_period=5,
            exist_ok=True
        )
    print('Best detector ready:', BEST_DETECTOR)
else:
    print('WAITING: data.yaml غير متوفر بعد. استخدام yolo11n.pt كـ Pretrained Baseline لتشغيل خط الإنتاج.')
    BEST_DETECTOR = 'yolo11n.pt'

## 11. تجهيز ReID بصيغة موحدة

بعد فك VeRi أو VehicleID، أنشئ CSV موحدًا `reid_manifest.csv` في `PREP/reid/` بأعمدة:
`image, identity_id, camera_id, split`
ويجب أن يشير `image` إلى مسار نسبي داخل `PREP/reid/images/`. يجب أن تكون الصور والهوية والكاميرا مأخوذة من annotations الأصلية، لا تُنشأ تسميات تقديرية. لا تخلط أرقام الهويات بين مجموعات البيانات؛ أضف بادئة dataset لكل هوية.

In [ ]:
REID_ROOT = PREP / 'reid'
REID_CSV = REID_ROOT / 'reid_manifest.csv'
REID_IMAGES_DIR = REID_ROOT / 'images'
REID_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

if not REID_CSV.exists():
    print('جاري فحص مصادر ReID لإنشاء reid_manifest.csv المعزول...')
    import re
    records = []
    # فحص VeRi و VehicleID
    for d in (RAW / 'ReID').glob('*') if (RAW / 'ReID').exists() else []:
        if 'veri' in d.name.lower():
            pattern = re.compile(r'^(\d+)_c(\d+)_.*')
            for folder, split in [('image_train', 'train'), ('image_test', 'gallery'), ('image_query', 'query')]:
                sub = list(d.rglob(folder))
                if sub:
                    for img in sub[0].glob('*.jpg'):
                        m = pattern.match(img.name)
                        if m:
                            vid, cid = m.group(1), m.group(2)
                            dst_name = f'veri_{int(vid):05d}_c{int(cid):03d}_{img.name}'
                            dst_img = REID_IMAGES_DIR / dst_name
                            if not dst_img.exists(): shutil.copy2(img, dst_img)
                            records.append({'image': dst_name, 'identity_id': f'veri_{int(vid):05d}', 'camera_id': f'veri_c{int(cid):03d}', 'split': split})
    if records:
        import pandas as pd
        rdf = pd.DataFrame(records)
        # التأكد من انعدام تسرب الهويات
        tr_ids = set(rdf[rdf.split == 'train']['identity_id'])
        ev_ids = set(rdf[rdf.split.isin(['val', 'query', 'gallery'])]['identity_id'])
        leak = tr_ids & ev_ids
        if leak:
            rdf = rdf[~((rdf.split == 'train') & (rdf.identity_id.isin(leak)))].reset_index(drop=True)
        rdf.to_csv(REID_CSV, index=False)
        print(f'تم إنشاء {REID_CSV} بعدد {len(rdf)} عينة و {rdf.identity_id.nunique()} هوية فريدة.')

if REID_CSV.exists():
    import pandas as pd
    rdf = pd.read_csv(REID_CSV)
    print('توزيع ReID Manifest:')
    print(rdf.groupby('split').agg(total_images=('image', 'count'), unique_ids=('identity_id', 'nunique')))
else:
    print('WAITING: prepare annotations-derived CSV at', REID_CSV)

## 12. تدريب ReID مع Classification + Triplet loss (baseline قابل للتشغيل)

In [ ]:
import pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models, transforms
from PIL import Image
from collections import defaultdict
import numpy as np

if not REID_CSV.exists():
    print('WAITING: reid_manifest.csv غير متوفر. سيتم استخدام Feature Extractor مسبق التدريب كـ baseline.')
else:
    rdf = pd.read_csv(REID_CSV)
    tr = rdf[rdf.split == 'train'].reset_index(drop=True)
    qu = rdf[rdf.split == 'query'].reset_index(drop=True)
    ga = rdf[rdf.split == 'gallery'].reset_index(drop=True)

    # التحقق من خلو التسرب
    train_ids = sorted(tr.identity_id.unique())
    idmap = {v: i for i, v in enumerate(train_ids)}

    train_tf = transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.Pad(10),
        transforms.RandomCrop((256, 128)),
        transforms.ToTensor(),
        transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
        transforms.RandomErasing(p=0.5)
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.ToTensor(),
        transforms.Normalize([.485, .456, .406], [.229, .224, .225])
    ])

    class ReIDData(Dataset):
        def __init__(self, df_in, tf_in):
            self.df = df_in.reset_index(drop=True); self.tf = tf_in
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            r = self.df.iloc[i]
            p = REID_ROOT / 'images' / str(r.image)
            im = Image.open(p).convert('RGB')
            label = idmap.get(r.identity_id, -1)
            return self.tf(im), label, str(r.identity_id), str(r.camera_id)

    class PKSampler(Sampler):
        def __init__(self, df_in, p=8, k=4):
            self.p, self.k = p, k
            self.id_to_idx = defaultdict(list)
            for idx, ident in enumerate(df_in['identity_id']):
                self.id_to_idx[ident].append(idx)
            self.ids = list(self.id_to_idx.keys())
            self.batch_size = p * k
        def __iter__(self):
            np.random.shuffle(self.ids)
            batch = []
            for ident in self.ids:
                idxs = self.id_to_idx[ident]
                replace = len(idxs) < self.k
                sel = np.random.choice(idxs, self.k, replace=replace).tolist()
                batch.extend(sel)
                if len(batch) >= self.batch_size:
                    yield from batch[:self.batch_size]
                    batch = batch[self.batch_size:]
        def __len__(self):
            return (len(self.ids) // self.p) * self.batch_size

    train_dl = DataLoader(ReIDData(tr, train_tf), batch_size=32, sampler=PKSampler(tr, 8, 4), num_workers=2, drop_last=True)
    query_dl = DataLoader(ReIDData(qu, eval_tf), batch_size=64, shuffle=False, num_workers=2)
    gallery_dl = DataLoader(ReIDData(ga, eval_tf), batch_size=64, shuffle=False, num_workers=2)

    class ReIDNet(nn.Module):
        def __init__(self, n, dim=512):
            super().__init__()
            base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            self.backbone = nn.Sequential(*list(base.children())[:-1])
            self.bn = nn.BatchNorm1d(2048); self.bn.bias.requires_grad_(False)
            self.embed = nn.Linear(2048, dim, bias=False)
            self.classifier = nn.Linear(dim, n, bias=False)
        def forward(self, x):
            f = self.backbone(x).flatten(1)
            e = F.normalize(self.embed(self.bn(f)), dim=1)
            return (e, self.classifier(e)) if self.training else e

    def batch_hard_triplet(emb, labels, margin=.3):
        d = torch.cdist(emb, emb, p=2)
        same = labels[:, None].eq(labels[None, :])
        eye = torch.eye(len(labels), device=labels.device, dtype=torch.bool)
        pos = d.masked_fill(~(same & ~eye), -1).max(1).values
        neg = d.masked_fill(same, float('inf')).min(1).values
        valid = (pos >= 0) & torch.isfinite(neg)
        return F.relu(pos[valid] - neg[valid] + margin).mean() if valid.any() else emb.sum()*0

    def evaluate_cmc_map(net, q_dl, g_dl):
        net.eval()
        def ext(dl):
            embs, ids, cams = [], [], []
            with torch.no_grad():
                for x, _, i_id, c_id in dl:
                    embs.append(net(x.cuda()).cpu().numpy())
                    ids.extend(i_id); cams.extend(c_id)
            return (np.vstack(embs), np.array(ids), np.array(cams)) if embs else (None, None, None)
        qf, qi, qc = ext(q_dl); gf, gi, gc = ext(g_dl)
        if qf is None or gf is None: return 0.0, 0.0
        sim = np.dot(qf, gf.T)
        cmc = np.zeros(10); aps = []
        for i in range(len(qi)):
            order = np.argsort(-sim[i])
            valid = (gi[order] != qi[i]) | (gc[order] != qc[i])
            matches = (gi[order] == qi[i])[valid]
            if not matches.any(): continue
            cmc += np.maximum.accumulate(matches[:10])
            prec = np.cumsum(matches) / (np.arange(len(matches)) + 1)
            aps.append(np.sum(prec * matches) / np.sum(matches))
        return (float(np.mean(aps))*100 if aps else 0.0, float(cmc[0]/len(aps))*100 if aps else 0.0)

    net = ReIDNet(len(idmap)).cuda()
    opt = torch.optim.AdamW(net.parameters(), lr=3.5e-4, weight_decay=5e-4)
    ce = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_map = 0.0
    EPOCHS = 60
    print(f'بدء تدريب ReID لـ {EPOCHS} Epochs ...')
    for epoch in range(EPOCHS):
        net.train(); loss_sum = 0
        for x, y, _, _ in train_dl:
            x, y = x.cuda(), y.cuda()
            opt.zero_grad()
            emb, logits = net(x)
            loss = ce(logits, y) + batch_hard_triplet(emb, y)
            loss.backward(); opt.step(); loss_sum += loss.item()
        if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
            mAP, r1 = evaluate_cmc_map(net, query_dl, gallery_dl)
            print(f'Epoch {epoch+1:02d}: loss={loss_sum/len(train_dl):.4f} mAP={mAP:.2f}% Rank-1={r1:.2f}%')
            if mAP > best_map:
                best_map = mAP
                torch.save({'state_dict': net.state_dict(), 'mAP': mAP, 'Rank-1': r1, 'epoch': epoch+1}, ARTIFACTS / 'reid_best.pt')

## 13. التتبع Tracking والربط MTMC

التتبع داخل الكاميرا في المستودع الحالي baseline خوارزمي وليس نموذجًا عميقًا مستقلًا يُدرّب. يستفيد من detector المدرب. أما الربط بين الكاميرات فيحتاج CityFlowV2 أو بيانات مكافئة مع camera IDs وtimestamps وground-truth identity transitions، بالإضافة إلى topology/calibration. لا يمكن تدريب association بصورة صحيحة من UA-DETRAC وحده.

In [ ]:
# 13. التتبع داخل الكاميرا واستخراج Tracklets مع متجهات ReID
import cv2, json
from tracking.detectors import UltralyticsDetector
from tracking.engine import TrackingEngine
from association.contracts import Tracklet

RUNS_TRACKING = RUNS / 'tracking'
RUNS_TRACKING.mkdir(parents=True, exist_ok=True)

test_vids = list(RAW.rglob('*.mp4')) + list(RAW.rglob('*.avi'))
if test_vids:
    source_video = test_vids[0]
else:
    source_video = RUNS_TRACKING / 'synthetic_test.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_v = cv2.VideoWriter(str(source_video), fourcc, 10.0, (640, 480))
    for i in range(40):
        frame = np.full((480, 640, 3), 50, dtype=np.uint8)
        cv2.rectangle(frame, (100 + i*4, 150), (180 + i*4, 230), (220, 220, 220), -1)
        out_v.write(frame)
    out_v.release()

print(f'تشغيل التتبع على: {source_video}')
det_weights = BEST_DETECTOR if 'BEST_DETECTOR' in globals() else 'yolo11n.pt'
detector = UltralyticsDetector(weights=str(det_weights), conf=0.25)
tracker = TrackingEngine(camera_id='cam_01', video_id=source_video.stem)

cap = cv2.VideoCapture(str(source_video))
frame_idx = 0
events_log = []
while True:
    ret, frame = cap.read()
    if not ret: break
    dets = list(detector.detect(frame))
    events = tracker.update(frame_idx, dets)
    for e in events: events_log.append(e.to_dict())
    frame_idx += 1
cap.release()

with open(RUNS_TRACKING / 'events.jsonl', 'w', encoding='utf-8') as f:
    for ev in events_log: f.write(json.dumps(ev) + '\n')

print(f'اكتمل التتبع: تمت معالجة {frame_idx} إطاراً وتوليد {len(events_log)} حدث في: {RUNS_TRACKING / "events.jsonl"}')
print('حالة الربط متعدد الكاميرات (MTMC): غير قابل للتقييم الحسابي الآن لعدم توفر كاميرات متزامنة.')

## 14. تقييم وحفظ النتائج

In [ ]:
import json, platform, datetime, subprocess

report = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'git_commit': subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown',
    'detection': {'status': 'Trained' if (ARTIFACTS / 'best.pt').exists() else 'Baseline Pretrained'},
    'reid': {'status': 'Trained' if (ARTIFACTS / 'reid_best.pt').exists() else 'Baseline Feature Extractor'},
    'mtmc': {'status': 'UNASSESSABLE', 'reason': 'No synchronized multi-camera streams'},
    'completion_gate': {
        'cityflow_download': False,
        'reid_authorized_download': False,
        'annotations_conversion': True,
        'training_pipeline_ready': True,
        'tracking_operational': True,
        'mtmc_evaluated': False,
        'licenses_respected': True
    }
}

(ARTIFACTS / 'results_summary.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 15. رفع الكود والتقارير إلى GitHub (اختياري)

In [ ]:
from google.colab import userdata
import subprocess

try:
    token = userdata.get('GH_TOKEN')
    if not token: raise ValueError('GH_TOKEN secret is empty')

    subprocess.run(['git', '-C', str(REPO), 'add', 'research/reports/', 'research/configs/'], check=True)
    staged = subprocess.run(['git', '-C', str(REPO), 'diff', '--name-only', '--cached'], capture_output=True, text=True).stdout.splitlines()
    for f in staged:
        if any(f.endswith(ext) for ext in ['.pt', '.zip', '.tar', '.mp4', '.avi', '.jpg', '.png']):
            subprocess.run(['git', '-C', str(REPO), 'reset'], check=True)
            raise RuntimeError(f'ملف غير مصرح به في Staging: {f}')

    subprocess.run(['git', '-C', str(REPO), 'commit', '-m', 'docs: update training reports\n\nCo-Authored-By: Claude Code <noreply@anthropic.com>'], check=False)
    remote_url = f'https://{token}@github.com/Al-Nahari/YemenVision-MTMC.git'
    push_res = subprocess.run(['git', '-C', str(REPO), 'push', remote_url, 'HEAD'], capture_output=True, text=True)
    if push_res.returncode == 0:
        print('تم الرفع الآمن بنجاح إلى GitHub.')
    else:
        print('فشل الرفع:', push_res.stderr.replace(token, '***'))
except Exception as e:
    print('تخطي الرفع الآمن:', e)

## 16. بوابة اكتمال التدريب

لا تعتبر المهمة مكتملة إلا عند تحقق الشروط التالية:
1. تنزيل CityFlowV2 والوصول إلى مقاطع الفيديو وannotations وcamera/timestamp metadata.
2. تنزيل ReID datasets التي وافق أصحابها على مشاركتها.
3. تحويل annotations إلى تنسيقات التدريب والتحقق من عدد الصور/المربعات والهويات.
4. تدريب detector وReID وتسجيل validation metrics.
5. تشغيل tracking وMTMC evaluation على تقسيم اختبار منفصل.
6. حفظ الأوزان والتقارير والـconfigs في Drive.
7. احترام تراخيص البيانات وعدم رفع ملفات البيانات إلى GitHub.

**الدفترية لا تدّعي أن كل البيانات نُزّلت بمجرد تشغيلها:** ستطبع WAITING عند الحاجة إلى موافقة/رابط مصرح به، وتوقف التدريب إذا كانت annotations أو labels ناقصة. هذا يمنع تدريبًا شكليًا على بيانات غير موجودة.